# Función para leer el set de datos

Aquí vamos a leer (Se pueden leer y modificar) datos almacenados en google drive.

En primer lugar construimos una función que permite cargar el set de imagenes de entrenamiento y validación junto con la categoria a la que pertenece cada una de estas imagenes.

In [5]:
import numpy as np
import os
import gzip

def cargar_set(ruta, tipo="train"):

  ruta_categorias = os.path.join(ruta, '%s-labels-idx1-ubyte.gz' %tipo)
  ruta_imagenes = os.path.join(ruta, '%s-images-idx3-ubyte.gz' %tipo)

  with gzip.open(ruta_categorias, 'rb') as rut_cat:
    etiquetas = np.frombuffer(rut_cat.read(), dtype=np.uint8, offset=8)

  with gzip.open(ruta_imagenes, 'rb') as rut_imgs:
    imagenes = np.frombuffer(rut_imgs.read(), dtype=np.uint8, offset=16).reshape(len(etiquetas),784)

  return imagenes, etiquetas

# Acceso a google drive

In [10]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

ruta = 'gdrive/My Drive/Notebooks GoogleColab/fashion_mnist_data'

X_train, Y_train = cargar_set(ruta, tipo='train')
X_test, Y_test = cargar_set(ruta, tipo='test')

Mounted at /content/gdrive


# Reajustar el tamaño de los datos

In [11]:
X_train = X_train[0:59904,:]
X_test = X_test[0:9984,:]
Y_train = Y_train[0:59904]
Y_test = Y_test[0:9984]

X_train = np.reshape(X_train,(59904,28,28,1))
X_test = np.reshape(X_test,(9984,28,28,1))

# Importar TensorFlow 2 (incluye Keras)

In [12]:
%tensorflow_version 2.x
import tensorflow as tf
print("Versión de tensor flow: " + tf.__version__)

Versión de tensor flow: 2.4.1


# Creación del modelo


In [16]:
tf.random.set_seed(200)

model = tf.keras.models.Sequential()

model.add(tf.keras.layers.BatchNormalization(input_shape=X_train.shape[1:]))
model.add(tf.keras.layers.Conv2D(64, (5, 5), padding='same', activation='elu'))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2,2)))
model.add(tf.keras.layers.Dropout(0.25))

model.add(tf.keras.layers.BatchNormalization(input_shape=X_train.shape[1:]))
model.add(tf.keras.layers.Conv2D(128, (5, 5), padding='same', activation='elu'))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2,2)))
model.add(tf.keras.layers.Dropout(0.25))

model.add(tf.keras.layers.BatchNormalization(input_shape=X_train.shape[1:]))
model.add(tf.keras.layers.Conv2D(256, (5, 5), padding='same', activation='elu'))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2,2)))
model.add(tf.keras.layers.Dropout(0.25))

model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(256))
model.add(tf.keras.layers.Activation('elu'))
model.add(tf.keras.layers.Dropout(0.5))
model.add(tf.keras.layers.Dense(10))
model.add(tf.keras.layers.Activation('softmax'))
model.summary()


Model: "sequential_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
batch_normalization_3 (Batch (None, 28, 28, 1)         4         
_________________________________________________________________
conv2d_3 (Conv2D)            (None, 28, 28, 64)        1664      
_________________________________________________________________
max_pooling2d_3 (MaxPooling2 (None, 14, 14, 64)        0         
_________________________________________________________________
dropout_4 (Dropout)          (None, 14, 14, 64)        0         
_________________________________________________________________
batch_normalization_4 (Batch (None, 14, 14, 64)        256       
_________________________________________________________________
conv2d_4 (Conv2D)            (None, 14, 14, 128)       204928    
_________________________________________________________________
max_pooling2d_4 (MaxPooling2 (None, 7, 7, 128)        

# Entrenamiento con CPU

In [17]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [20]:
import timeit

def entrenamiento_cpu():
  with tf.device('/cpu:0'):
    model.fit(X_train,Y_train,validation_data=(X_test,Y_test),batch_size=128,epochs=2,verbose=1)
  
  return None

cpu_time = timeit.timeit('entrenamiento_cpu()', number=1, setup='from __main__ import entrenamiento_cpu')

Epoch 1/2
468/468 [==============================] - 726s 2s/step - loss: 0.6430 - accuracy: 0.7895 - val_loss: 0.3931 - val_accuracy: 0.8660
Epoch 2/2
468/468 [==============================] - 727s 2s/step - loss: 0.3902 - accuracy: 0.8627 - val_loss: 0.3288 - val_accuracy: 0.8851


In [21]:
print('Tiempo de entrenamiento: ' + str(cpu_time) + ' segundos')

Tiempo de entrenamiento: 1452.919399864999 segundos
